In [75]:
import torch
import tiktoken
from torch import nn

In [76]:
torch.manual_seed(123)

In [77]:
# Hyperparameters
config = {
    "context_length": 6,
    "dim_in": 4,
    "dim_out": 3,
    "dropout_rate": 0.1
}

dim_in = 4
dim_out = 3
context_length = 6

In [78]:
class TokenizerV1:
    def __init__(self):
        self.tokenizer = tiktoken.get_encoding("gpt2")
        self.n_vocab = self.tokenizer.n_vocab

    def encode(self, text):
        return self.tokenizer.encode(text)

    def decode(self, encodings):
        return self.tokenizer.decode(encodings)

In [79]:
class InputPreprocessor(nn.Module):
    def __init__(self, tokenizer):
        super().__init__()
        self.tokenizer = tokenizer
        self.token_embeddings = nn.Embedding(tokenizer.n_vocab, config["dim_in"])
        self.pos_embeddings = nn.Embedding(config["context_length"], config["dim_in"])

    def forward(self, text):
        encodings = self.tokenizer.encode(text)
        encodings = torch.tensor(encodings[:config["context_length"]])
        return self.token_embeddings(encodings) + self.pos_embeddings(torch.arange(0, config["context_length"]))
        

In [80]:
tokenizer = TokenizerV1()
preprocessor = InputPreprocessor(tokenizer)

In [81]:
input_embeddings = preprocessor("I want to play football.")

In [82]:
W_keys = nn.Linear(dim_in, dim_out)
W_queries = nn.Linear(dim_in, dim_out)
W_values = nn.Linear(dim_in, dim_out)

In [83]:
keys = W_keys(input_embeddings)
queries = W_queries(input_embeddings)
values = W_values(input_embeddings)
d_k = keys.shape[-1]

In [84]:
attention_scores = queries @ keys.T 
attention_weights = torch.softmax(attention_scores / (d_k ** 0.5), dim=-1) # attention weights
attention_weights

tensor([[0.0838, 0.3415, 0.0856, 0.0495, 0.2554, 0.1842],
        [0.1661, 0.0301, 0.2424, 0.4076, 0.0636, 0.0902],
        [0.1100, 0.2533, 0.1236, 0.1121, 0.2182, 0.1828],
        [0.0739, 0.3147, 0.0842, 0.0282, 0.2992, 0.1998],
        [0.1632, 0.1227, 0.1918, 0.2127, 0.1508, 0.1589],
        [0.1406, 0.1897, 0.1566, 0.1331, 0.1978, 0.1821]],
       grad_fn=<SoftmaxBackward0>)

In [85]:
mask = torch.tril(torch.ones(context_length, context_length))
mask

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])

In [86]:
attention_weights * mask

tensor([[0.0838, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1661, 0.0301, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1100, 0.2533, 0.1236, 0.0000, 0.0000, 0.0000],
        [0.0739, 0.3147, 0.0842, 0.0282, 0.0000, 0.0000],
        [0.1632, 0.1227, 0.1918, 0.2127, 0.1508, 0.0000],
        [0.1406, 0.1897, 0.1566, 0.1331, 0.1978, 0.1821]],
       grad_fn=<MulBackward0>)

In [87]:
(attention_weights * mask) / torch.sum(attention_weights * mask, dim=-1, keepdim=True)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.8467, 0.1533, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2260, 0.5203, 0.2537, 0.0000, 0.0000, 0.0000],
        [0.1475, 0.6282, 0.1680, 0.0563, 0.0000, 0.0000],
        [0.1940, 0.1459, 0.2280, 0.2529, 0.1793, 0.0000],
        [0.1406, 0.1897, 0.1566, 0.1331, 0.1978, 0.1821]],
       grad_fn=<DivBackward0>)

In [88]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
mask.bool()

tensor([[False,  True,  True,  True,  True,  True],
        [False, False,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True],
        [False, False, False, False,  True,  True],
        [False, False, False, False, False,  True],
        [False, False, False, False, False, False]])

In [89]:
attention_scores

tensor([[-0.4681,  1.9656, -0.4303, -1.3786,  1.4623,  0.8969],
        [-0.1567, -3.1167,  0.4975,  1.3980, -1.8190, -1.2151],
        [-0.5642,  0.8800, -0.3636, -0.5320,  0.6212,  0.3152],
        [-0.4883,  2.0215, -0.2630, -2.1553,  1.9336,  1.2343],
        [-0.2234, -0.7167,  0.0564,  0.2360, -0.3600, -0.2697],
        [-0.2813,  0.2373, -0.0948, -0.3769,  0.3093,  0.1658]],
       grad_fn=<MmBackward0>)

In [90]:
masked_score = attention_scores.masked_fill(mask.bool(), -torch.inf)
masked_score

tensor([[-0.4681,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.1567, -3.1167,    -inf,    -inf,    -inf,    -inf],
        [-0.5642,  0.8800, -0.3636,    -inf,    -inf,    -inf],
        [-0.4883,  2.0215, -0.2630, -2.1553,    -inf,    -inf],
        [-0.2234, -0.7167,  0.0564,  0.2360, -0.3600,    -inf],
        [-0.2813,  0.2373, -0.0948, -0.3769,  0.3093,  0.1658]],
       grad_fn=<MaskedFillBackward0>)

In [91]:
attn_weights = torch.softmax(masked_score / d_k ** 0.5, dim=-1)
attn_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.8467, 0.1533, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2260, 0.5203, 0.2537, 0.0000, 0.0000, 0.0000],
        [0.1475, 0.6282, 0.1680, 0.0563, 0.0000, 0.0000],
        [0.1940, 0.1459, 0.2280, 0.2529, 0.1793, 0.0000],
        [0.1406, 0.1897, 0.1566, 0.1331, 0.1978, 0.1821]],
       grad_fn=<SoftmaxBackward0>)

In [92]:
dropout = nn.Dropout(0.1)

In [93]:
attn_weights = dropout(attn_weights)
attn_weights

tensor([[1.1111, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.9408, 0.1703, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2511, 0.5781, 0.2819, 0.0000, 0.0000, 0.0000],
        [0.1639, 0.6980, 0.1866, 0.0626, 0.0000, 0.0000],
        [0.2155, 0.1621, 0.2533, 0.2810, 0.1992, 0.0000],
        [0.1563, 0.2108, 0.1740, 0.1479, 0.2198, 0.2023]],
       grad_fn=<MulBackward0>)

In [94]:
attn_weights @ values

tensor([[-1.7024,  0.4932, -0.6789],
        [-1.1263,  0.1495, -0.4793],
        [ 0.4988, -0.9106, -0.0428],
        [ 0.9269, -1.1158,  0.2538],
        [-0.1388, -0.2744,  0.2498],
        [-0.0099, -0.1984,  0.1665]], grad_fn=<MmBackward0>)

In [95]:
class CausalAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.context_length = config["context_length"]
        self.dim_in = config["dim_in"]
        self.dim_out = config["dim_out"]
        self.W_key = nn.Linear(self.dim_in, self.dim_out)
        self.W_query = nn.Linear(self.dim_in, self.dim_out)
        self.W_value = nn.Linear(self.dim_in, self.dim_out)
        self.dropout = nn.Dropout(config["dropout_rate"])
        self.mask = torch.triu(torch.ones(self.context_length, self.context_length), diagonal=1).bool()

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attention_scores = queries @ keys.transpose(1,2)
        attention_scores.masked_fill_(self.mask, -torch.inf)
        attention_weights = torch.softmax(attention_scores / (keys.shape[-1] ** 0.5), dim=-1)
        attention_weights = self.dropout(attention_weights)
        context_vectors = attention_weights @ values
        return context_vectors

In [96]:
inputs = input_embeddings

In [97]:
inputs

tensor([[-0.6548,  2.1284, -0.2805,  1.3280],
        [ 0.9075, -0.3510, -0.6833, -2.9320],
        [ 0.3414,  0.5612, -0.7663,  1.4225],
        [ 2.4883, -0.4090,  2.2379,  0.2385],
        [-0.9753, -0.0234,  0.6451, -1.2915],
        [-0.9374,  0.3967,  0.8233, -0.6069]], grad_fn=<AddBackward0>)

In [98]:
ca = CausalAttention()

In [99]:
torch.unsqueeze(inputs, dim=0).shape

torch.Size([1, 6, 4])

In [27]:
ca(torch.unsqueeze(inputs, dim=0))

tensor([[[ 0.5937, -0.0557,  0.9900],
         [-0.1793, -0.6526, -0.0965],
         [-0.0158, -0.1750,  0.5495],
         [-0.3524,  0.1624,  1.0333],
         [-0.5838,  0.0749,  0.6810],
         [-0.5644, -0.2910,  0.5058]]], grad_fn=<UnsafeViewBackward0>)

In [47]:
ones = torch.ones(3,4)
twos = torch.ones(3,4) * 2

In [63]:
batches = torch.ones(2,12)

In [64]:
batches.shape

torch.Size([2, 12])

In [65]:
batches.view(2,3,4)

tensor([[[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]],

        [[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]]])

In [51]:
trans1 = batches.view(2,3,2,2).transpose(1,2)
trans1

tensor([[[[1., 1.],
          [1., 1.],
          [1., 1.]],

         [[1., 1.],
          [1., 1.],
          [1., 1.]]],


        [[[2., 2.],
          [2., 2.],
          [2., 2.]],

         [[2., 2.],
          [2., 2.],
          [2., 2.]]]])

In [55]:
trans1.shape

torch.Size([2, 2, 3, 2])

In [56]:
trans1.transpose(0,1).shape

torch.Size([2, 2, 3, 2])

In [57]:
trans1.transpose(0,1)

tensor([[[[1., 1.],
          [1., 1.],
          [1., 1.]],

         [[2., 2.],
          [2., 2.],
          [2., 2.]]],


        [[[1., 1.],
          [1., 1.],
          [1., 1.]],

         [[2., 2.],
          [2., 2.],
          [2., 2.]]]])

In [58]:
trans1 @ trans1.transpose(2,3)

tensor([[[[2., 2., 2.],
          [2., 2., 2.],
          [2., 2., 2.]],

         [[2., 2., 2.],
          [2., 2., 2.],
          [2., 2., 2.]]],


        [[[8., 8., 8.],
          [8., 8., 8.],
          [8., 8., 8.]],

         [[8., 8., 8.],
          [8., 8., 8.],
          [8., 8., 8.]]]])